# Transformers, Day 1 — Companion Notebook
## Debriefing *Attention Is All You Need*

**COMP 395 — Deep Learning**

Keep your **Assignment #4** sheet next to you. Every part of this notebook starts from the numbers on it.

| Part | Paper section | What you do |
|---|---|---|
| 1 | 3.2.1 | Check your Assignment #4 answers in code, then see why the paper divides by $\sqrt{d_k}$ |
| 2 | 3.2.2 | Why project at all? Then a multi-head shape trace |
| 3 | 3.2.3 | Mask the decoder so it can't peek |
| 4 | 3.5 | Rearrange `not very good` and see that attention ignores word order |
| 5 | 3.5 | Add positional encoding and see that the order now matters |
| 6 | 3.2 | Build `SelfAttention(nn.Module)` and verify it against Assignment #4 |

**Keep the training task in mind:** the model is only ever asked to predict the next word. Every matrix below ($W_Q$, $W_K$, $W_V$, one set per head) is something that training loop tunes. On Assignment #4 they were given; here you will see what they make possible.

**All the code is written for you.** Run the cells in order. Cells marked **Try** ask you to change one thing (a word order, a number) and run again. Answer the questions as you go.

## Setup

In [ ]:
import math
import os
import zipfile
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(395)
torch.set_printoptions(precision=2, sci_mode=False)

def plot_attention(weights, words, title="Attention weights", ax=None):
    '''Heatmap of a (n, n) attention matrix. Rows = the word asking, columns = the word looked at.'''
    w = weights.detach()
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(w, cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=45, ha="right")
    ax.set_yticks(range(len(words))); ax.set_yticklabels(words)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center",
                    color="white" if w[i, j] > 0.6 else "black", fontsize=9)
    ax.set_xlabel("looked at"); ax.set_ylabel("asking"); ax.set_title(title)
    return ax

### GloVe vectors (used in Part 6)

Part 6 uses real 50-dimensional GloVe word vectors for a full sentence. The loader below looks for the `glove.6B.zip` you downloaded for the Day 2 assignment (in `./data`), and otherwise downloads it. If no GloVe can be found it falls back to random vectors so the notebook still runs, but the attention patterns will then be meaningless.

In [ ]:
def load_glove50(words):
    '''Return a (len(words), 50) tensor of GloVe-50d vectors for `words` (lowercased).'''
    candidates = ["./data/glove.6B.zip", "./glove.6B.zip", "../data/glove.6B.zip"]
    zip_path = next((p for p in candidates if os.path.exists(p)), None)
    if zip_path is None:
        zip_path = "./data/glove.6B.zip"
        os.makedirs("./data", exist_ok=True)
        try:
            print("Downloading GloVe (822 MB, once)...")
            urllib.request.urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip", zip_path)
        except Exception as e:
            print("Could not download GloVe:", e)
            print("Falling back to RANDOM vectors. Attention patterns below will not mean anything.")
            return torch.randn(len(words), 50)
    wanted = {w.lower() for w in words}
    vecs = {}
    with zipfile.ZipFile(zip_path) as z, z.open("glove.6B.50d.txt") as f:
        for line in f:
            parts = line.decode("utf-8").split()
            if parts[0] in wanted:
                vecs[parts[0]] = torch.tensor([float(x) for x in parts[1:]])
                if len(vecs) == len(wanted):
                    break
    out = []
    for w in words:
        if w.lower() not in vecs:
            print(f"'{w}' not in GloVe, using a random vector")
        out.append(vecs.get(w.lower(), torch.randn(50)))
    return torch.stack(out)

---
## Part 1 — Assignment #4 in code, and why $\sqrt{d_k}$ (paper §3.2.1)

Here are the matrices from your sheet.

In [ ]:
words = ["not", "very", "good"]

X   = torch.tensor([[-1., 1], [-2, 1], [2, -1]])     # one row per word
W_Q = torch.tensor([[-1., -1], [0, 1]])
W_K = torch.tensor([[-1., 0], [-1, -1]])
W_V = torch.tensor([[0., -1], [-1, 1]])

**Run this:** the four steps from your sheet, in PyTorch. `@` is matrix multiplication, `.T` is transpose, and `softmax(..., dim=-1)` works on each row.

In [ ]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
S = Q @ K.T
A = torch.softmax(S, dim=-1)
Z = A @ V

print("S =\n", S)
print("A =\n", A)
print("Z =\n", Z)

In [ ]:
# Check against the sheet (your hand values may differ by ~0.01 because the e^x table was rounded)
S_sheet = torch.tensor([[-2., -1, 1], [-3, -1, 1], [3, 1, -1]])
A_sheet = torch.tensor([[.04, .12, .84], [.02, .12, .87], [.87, .12, .02]])
print("Matches Assignment #4:", bool(torch.allclose(A, A_sheet, atol=0.02)))
plot_attention(A, words, "Assignment #4"); plt.show()

### Why divide by $\sqrt{d_k}$?

The paper (§3.2.1): *"for large values of $d_k$, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients."*

Your scores are small because $d_k = 2$. Pretend they were bigger.

**Try:** add more numbers to `scales` (say 20 or 0.5) and run again.

In [ ]:
scales = [1, 3, 10]
for scale in scales:
    print(f"scores x {scale:2d}  ->  softmax of row 'not':", torch.softmax(S[0] * scale, dim=-1))

**Question 1.2** — As the scores grow, what happens to the weights for `not`? Why is that bad for *learning* (think about what the gradient of a function that has flattened out looks like, and what happened to $\tanh$ on Day 1)?

*Your answer:*

In [ ]:
d_k = 2
A_scaled = torch.softmax(S / math.sqrt(d_k), dim=-1)
print("With the paper's scaling, row 'not':", A_scaled[0])

---
## Part 2 — Linear projections and heads (paper §3.2.2)

**Why project at all?** Try attention with no projections, $Q = K = V = X$, on your sheet's words, and compare with your Assignment #4 weights.

In [ ]:
A_noproj = torch.softmax(X @ X.T, dim=-1)     # no W_Q, W_K, W_V

print("row 'very', no projections:  ", A_noproj[1])
print("row 'very', with W_Q and W_K:", A[1])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_attention(A_noproj, words, "no projections (Q = K = V = X)", axes[0])
plot_attention(A, words, "with W_Q, W_K (Assignment #4)", axes[1])
plt.tight_layout(); plt.show()

**Question 2.1** — Without projections, where does `very` look, and how much weight does it give `good`? Use $x_{\texttt{very}} \cdot x_{\texttt{good}}$ to explain why. What do the learned $W_Q, W_K$ make possible?

*Your answer:*

### Multi-head shape trace

$n = 10$ tokens, $d_{\text{model}} = 128$, $h = 8$ heads, so each head works in $d_k = d_{\text{model}} / h$ dimensions.

**Run this**, then **Try:** change `h` to 4 or 16 and run again. Which shapes change?

In [ ]:
n, d_model, h = 10, 128, 8
d_k = d_model // h

shapes = {
    "X":                     (n, d_model),
    "Q_i for one head":      (n, d_k),
    "scores for one head":   (n, n),
    "head_i output":         (n, d_k),
    "concat of all heads":   (n, h * d_k),
    "after W^O":             (n, d_model),
}
for name, shape in shapes.items():
    print(f"{name:24s} {shape}")

**Question 2.3** — If we use $h = 4$ heads instead (same $d_{\text{model}}$), what is $d_k$? Does the total number of parameters in $W^Q, W^K, W^V$ across all heads change?

*Your answer:*

---
## Part 3 — Masking: don't peek (paper §3.2.3)

In the decoder, all Spanish words are processed at once (that is the whole point of dropping the RNN). But when the model predicts word 2 it must not look at words 3, 4, … or it would just copy the answer.

The paper's fix: before softmax, set every score to the **right** of the diagonal to $-\infty$. Since $e^{-\infty} = 0$, those words get weight exactly 0.

**Run this.** `torch.triu(..., diagonal=1)` puts 1s above the diagonal, and `masked_fill` turns those scores into $-\infty$.

In [ ]:
n = len(words)
mask = torch.triu(torch.ones(n, n), diagonal=1)
S_masked = S.masked_fill(mask.bool(), float("-inf"))
A_masked = torch.softmax(S_masked, dim=-1)

print("masked scores =\n", S_masked)
print("masked weights =\n", A_masked)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_attention(A, words, "unmasked (Assignment #4)", axes[0])
plot_attention(A_masked, words, "masked", axes[1])
plt.tight_layout(); plt.show()

**Question 3.2** — Which row is unchanged from Assignment #4, and why? Which row changed the most?

*Your answer:*

**Question 3.3** — In your seq2seq training script the decoder was fed the correct previous word at every step (teacher forcing). How does the mask let the Transformer do all steps of teacher forcing *at once*?

*Your answer:*

---
## Part 4 — Attention ignores word order (paper §3.5)

This is your Assignment #4 attention, with your $W_Q$, $W_K$, $W_V$. Each word keeps its own 2 numbers from the sheet. What happens if we put the words in a different order?

In [ ]:
def self_attention(X):
    '''Assignment #4, as a function: X -> (Z, A).'''
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    A = torch.softmax(Q @ K.T, dim=-1)
    return A @ V, A

emb = {w: X[i] for i, w in enumerate(words)}      # not, very, good -> their 2 numbers
fmt = lambda z: "[" + ", ".join(f"{v:6.2f}" for v in z.tolist()) + "]"   # print 2 decimals
Z_orig, A_orig = self_attention(X)

**Prediction (write before running):** if the phrase is `good not very` instead, will `good`'s output change?

*Your prediction:*

**Try:** rearrange the words in `scrambled` any way you like, and run again.

In [ ]:
scrambled = ["good", "not", "very"]     # <- rearrange me

assert sorted(scrambled) == sorted(words), "use the same three words"
X_scr = torch.stack([emb[w] for w in scrambled])
Z_scr, A_scr = self_attention(X_scr)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_attention(A_orig, words, " ".join(words), axes[0])
plot_attention(A_scr, scrambled, " ".join(scrambled), axes[1])
plt.tight_layout(); plt.show()

print("each word's output Z, in both orders:")
for w in words:
    print(f"  {w:5s}  {fmt(Z_orig[words.index(w)])}   vs   {fmt(Z_scr[scrambled.index(w)])}")

**Question 4.1** — Every word got exactly the same output in both orders. So "dog bites man" and "man bites dog" would look the same to attention. In one sentence, why? What is attention missing?

*Your answer:*

---
## Part 5 — Sinusoidal positional encoding (paper §3.5)

The fix: add a position vector to each embedding **before** attention, $\tilde{x}_i = x_i + p_i$. The paper uses

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

**Run this:** the paper's formula in three lines. `freq` holds $1/10000^{2i/d}$ for each pair of columns, and `position * freq` gives every angle at once.

In [ ]:
def positional_encoding(max_len, d):
    PE = torch.zeros(max_len, d)
    position = torch.arange(max_len).unsqueeze(1).float()      # shape (max_len, 1)
    i = torch.arange(0, d, 2).float()                           # 0, 2, 4, ...  (this is 2i)
    freq = 1.0 / 10000 ** (i / d)                               # shape (d/2,)
    PE[:, 0::2] = torch.sin(position * freq)                    # even columns
    PE[:, 1::2] = torch.cos(position * freq)                    # odd columns
    return PE

PE = positional_encoding(50, 64)          # 50 positions, 64 numbers each

# Each column of PE is a wave over positions. Plot three of them.
plt.figure(figsize=(9, 4))
for col, label in [(0, "column 0 (fast)"), (10, "column 10 (medium)"), (30, "column 30 (slow)")]:
    plt.plot(PE[:, col], "o-", markersize=3, label=label)
plt.xlabel("position in the sentence"); plt.ylabel("value added to the word")
plt.title("Three of the 64 numbers in each position's vector")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print("position 0 :", PE[0, [0, 10, 30]].numpy().round(2))
print("position 1 :", PE[1, [0, 10, 30]].numpy().round(2))
print("position 6 :", PE[6, [0, 10, 30]].numpy().round(2))
print("position 40:", PE[40, [0, 10, 30]].numpy().round(2))

**Question 5.2** — Look at positions 0 and 6 in the printout. Which column can barely tell them apart, and which one can? Why does the model need both fast and slow waves?

*Your answer:*

Now give your three words positions. With 2 numbers per word, position $p$ gets the vector $[\sin p,\ \cos p]$. Then rerun **your** `scrambled` order from Part 4.

In [ ]:
P = positional_encoding(3, 2)        # one row per position: 1st, 2nd, 3rd
print("position vectors:\n", P)

Z_orig_pe, A_orig_pe = self_attention(X + P)          # word i sits at position i
Z_scr_pe,  A_scr_pe  = self_attention(X_scr + P)      # same words, new positions

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_attention(A_orig_pe, words, " ".join(words) + "  (+ positions)", axes[0])
plot_attention(A_scr_pe, scrambled, " ".join(scrambled) + "  (+ positions)", axes[1])
plt.tight_layout(); plt.show()

print("each word's output Z, in both orders, with positions:")
for w in words:
    print(f"  {w:5s}  {fmt(Z_orig_pe[words.index(w)])}   vs   {fmt(Z_scr_pe[scrambled.index(w)])}")

**Question 5.3** — Now the outputs differ between the two orders. What changed about the *inputs to the dot product* that made this happen?

*Your answer:*

---
## Part 6 — `SelfAttention` as an `nn.Module` (paper §3.2)

`nn.Linear(d_model, d_k, bias=False)` **is** a weight matrix like $W_Q$ on your sheet, except PyTorch learns it.

**Run this:** one head of attention as a PyTorch module. Read it line by line against your sheet. The shapes, for input `X` of shape `(n, d_model)`:

| | shape |
|---|---|
| `Q`, `K`, `V` | `(n, d_k)` |
| `scores` | `(n, n)` |
| `weights` | `(n, n)` |
| `output` | `(n, d_k)` |

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.scale = d_k ** 0.5

    def forward(self, X):
        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X)
        scores = Q @ K.T / self.scale
        weights = torch.softmax(scores, dim=-1)
        output = weights @ V
        return output, weights

### Test 1: it must reproduce Assignment #4

We load your sheet's matrices into the module. (`nn.Linear` stores the matrix transposed, hence `.T`.) Your sheet skipped the $\sqrt{d_k}$, so we set `scale = 1` for this test.

In [ ]:
sa = SelfAttention(d_model=2, d_k=2)
with torch.no_grad():
    sa.W_Q.weight.copy_(W_Q.T)
    sa.W_K.weight.copy_(W_K.T)
    sa.W_V.weight.copy_(W_V.T)
sa.scale = 1.0
Z_mod, A_mod = sa(X)
print("A from the module =\n", A_mod)
print("Z from the module =\n", Z_mod)
print("✓ matches Assignment #4" if torch.allclose(A_mod, A_sheet, atol=0.02) else "✗ does not match")

### Test 2: a real sentence, random (untrained) weights

**Try:** replace `chicken` with a sentence of your own (lowercase words, as a list) and run again.

In [ ]:
chicken = ["the", "chicken", "did", "not", "cross", "the", "road", "because", "it"]   # <- try your own
E_ch = load_glove50(chicken)

torch.manual_seed(395)
sa = SelfAttention(d_model=50, d_k=25)
_, A_ch    = sa(E_ch)
_, A_ch_pe = sa(E_ch + positional_encoding(len(chicken), 50))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_attention(A_ch, chicken, "untrained, no PE", axes[0])
plot_attention(A_ch_pe, chicken, "untrained, with PE", axes[1])
plt.tight_layout(); plt.show()

**Question 6.2** — These weights come from random $W_Q, W_K, W_V$. Should the row for `it` look meaningful yet? After training on enough text, what would you expect that row to look like?

*Your answer:*

**Question 6.3** — The paper runs 8 of these modules side by side (multi-head). Give one reason, using the chicken sentence, why one module isn't enough.

*Your answer:*

---
## Exit ticket (Canvas)

Draw a concept map linking: *self-attention · multi-head attention · positional encoding · masking · query · key · value · softmax · $\sqrt{d_k}$ · teacher forcing · cross-attention · BLEU · vanishing gradient*. Label each link with a short phrase.